In [ ]:
pip install transformers datasets accelerate bitsandbytes peft


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "openlm-research/open_llama_3b"  # Lightweight open LLaMA model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", load_in_4bit=True)


In [ ]:
from datasets import load_dataset

dataset = load_dataset("tiny_shakespeare")  # ~1MB
dataset = dataset["train"].train_test_split(test_size=0.1)


In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(tokenize, batched=True)


In [ ]:
from peft import get_peft_model, LoraConfig, TaskType
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    output_dir="./llama-shakespeare",
    evaluation_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_dir="./logs",
    fp16=True,
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()


In [ ]:
prompt = "To be, or not to be: that is the question"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=100, do_sample=True, top_k=50, temperature=0.9)
print(tokenizer.decode(output[0], skip_special_tokens=True))


In [ ]:
# ============================================================
# WEEK 9 - FAST OPEN LLaMA TEXT GENERATION
# ============================================================

!pip install -q transformers datasets accelerate peft sentencepiece

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

# ------------------------------------------------------------
# Load OpenLLaMA
# ------------------------------------------------------------

model_name = "openlm-research/open_llama_3b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available()
    else torch.float32
)

# ------------------------------------------------------------
# Load Shakespeare Dataset
# ------------------------------------------------------------

dataset = load_dataset(
    "text",
    data_files={"train": "shakespeare.txt"}
)

# Use only first 500 samples
dataset["train"] = dataset["train"].select(
    range(min(500, len(dataset["train"])))
)

dataset = dataset["train"].train_test_split(test_size=0.1)

# ------------------------------------------------------------
# Tokenization
# ------------------------------------------------------------

def tokenize_function(examples):

    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# ------------------------------------------------------------
# LoRA Configuration
# ------------------------------------------------------------

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, lora_config)

# ------------------------------------------------------------
# Data Collator
# ------------------------------------------------------------

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# ------------------------------------------------------------
# Training Arguments
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./open_llama_output",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    max_steps=50,
    logging_steps=10,
    fp16=torch.cuda.is_available()
)

# ------------------------------------------------------------
# Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

trainer.train()

# ------------------------------------------------------------
# Generate Text
# ------------------------------------------------------------

prompt = "To be, or not to be"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

if torch.cuda.is_available():
    model = model.cuda()
    inputs = {k: v.cuda() for k, v in inputs.items()}

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50
)

print("\nGenerated Text:\n")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/237 [00:00<?, ?it/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Step,Training Loss
10,6.497099
20,6.215480
30,6.040967
40,6.287116
50,5.980786



Generated Text:

To be, or not to be, that is the question — what is the purpose of life, if not to live?
There are many answers to this question. For some people, the purpose of life can be found in the pursuit of money or fame. Others take a more
